In [1]:
rdd = spark.sparkContext.textFile("s3a://moviereviewpractice/ml-32m/ratings.csv")

In [2]:
rdd.first()

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
                                                                                

'userId,movieId,rating,timestamp'

In [3]:
def is_valid_rating(line):
    try:
        float(line[2])
        return True
    except:
        return False

In [4]:
raw_rdd = rdd.map(lambda line: line.split(","))
split_rdd = raw_rdd.filter(lambda line: len(line) == 4)
clean_rdd = split_rdd.filter(is_valid_rating)
movie_rdd = clean_rdd.map(lambda line: (int(line[1]), float(line[2])))

In [5]:
movie_rdd.take(5)

[(17, 4.0), (25, 1.0), (29, 2.0), (30, 5.0), (32, 5.0)]

In [6]:
movie_rdd_transformed = movie_rdd.map(lambda line: (line[0], (line[1], 1)))

In [7]:
movie_rdd_transformed.first()

(17, (4.0, 1))

In [8]:
movie_rdd_reduced = movie_rdd_transformed.reduceByKey(lambda record1, record2:(record1[0] + record2[0], record1[1] + record2[1])) 

In [9]:
movie_rdd_reduced = movie_rdd_reduced.filter(lambda record: record[1][1] > 100)

In [10]:
movie_rdd_average = movie_rdd_reduced.map(lambda line: (line[0], line[1][0] / line[1][1]))

In [11]:
movie_rdd_sorted = movie_rdd_average.sortBy(lambda line: line[1], False)

In [12]:
movie_rdd_sorted.take(10)

[(171011, 4.4468302658486705),
 (159817, 4.444369063772049),
 (170705, 4.426538598363572),
 (318, 4.404613860039444),
 (171495, 4.330081300813008),
 (858, 4.317030403371463),
 (202439, 4.312253641816624),
 (179135, 4.300085984522786),
 (198185, 4.298684210526316),
 (220528, 4.28619153674833)]

In [13]:
import csv

In [14]:
movie_info = spark.sparkContext.textFile("s3a://moviereviewpractice/ml-32m/movies.csv")

In [15]:
movie_info.take(5)

['movieId,title,genres',
 '1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy',
 '2,Jumanji (1995),Adventure|Children|Fantasy',
 '3,Grumpier Old Men (1995),Comedy|Romance',
 '4,Waiting to Exhale (1995),Comedy|Drama|Romance']

In [16]:
def parse_movie(line):
    return next(csv.reader([line]))

In [17]:
movie_info = movie_info.map(lambda line: parse_movie(line))

In [18]:
movie_info.take(100)

[['movieId', 'title', 'genres'],
 ['1', 'Toy Story (1995)', 'Adventure|Animation|Children|Comedy|Fantasy'],
 ['2', 'Jumanji (1995)', 'Adventure|Children|Fantasy'],
 ['3', 'Grumpier Old Men (1995)', 'Comedy|Romance'],
 ['4', 'Waiting to Exhale (1995)', 'Comedy|Drama|Romance'],
 ['5', 'Father of the Bride Part II (1995)', 'Comedy'],
 ['6', 'Heat (1995)', 'Action|Crime|Thriller'],
 ['7', 'Sabrina (1995)', 'Comedy|Romance'],
 ['8', 'Tom and Huck (1995)', 'Adventure|Children'],
 ['9', 'Sudden Death (1995)', 'Action'],
 ['10', 'GoldenEye (1995)', 'Action|Adventure|Thriller'],
 ['11', 'American President, The (1995)', 'Comedy|Drama|Romance'],
 ['12', 'Dracula: Dead and Loving It (1995)', 'Comedy|Horror'],
 ['13', 'Balto (1995)', 'Adventure|Animation|Children'],
 ['14', 'Nixon (1995)', 'Drama'],
 ['15', 'Cutthroat Island (1995)', 'Action|Adventure|Romance'],
 ['16', 'Casino (1995)', 'Crime|Drama'],
 ['17', 'Sense and Sensibility (1995)', 'Drama|Romance'],
 ['18', 'Four Rooms (1995)', 'Comedy']

In [19]:
movie_info = movie_info.zipWithIndex().filter(lambda x: x[1] > 0).map(lambda x: x[0])

In [20]:
movie_info.first()

['1', 'Toy Story (1995)', 'Adventure|Animation|Children|Comedy|Fantasy']

In [21]:
movie_info_transformed = movie_info.map(lambda line: (int(line[0]), line[1]))

In [22]:
movie_info_transformed.take(10)

[(1, 'Toy Story (1995)'),
 (2, 'Jumanji (1995)'),
 (3, 'Grumpier Old Men (1995)'),
 (4, 'Waiting to Exhale (1995)'),
 (5, 'Father of the Bride Part II (1995)'),
 (6, 'Heat (1995)'),
 (7, 'Sabrina (1995)'),
 (8, 'Tom and Huck (1995)'),
 (9, 'Sudden Death (1995)'),
 (10, 'GoldenEye (1995)')]

In [23]:
movie_rdd = movie_rdd_sorted.join(movie_info_transformed)

In [24]:
movie_rdd.take(5)

[(286897, (4.207413945278023, 'Spider-Man: Across the Spider-Verse (2023)')),
 (160718, (4.15354535974974, 'Piper (2016)')),
 (1131, (4.103291713961408, 'Jean de Florette (1986)')),
 (1189, (4.09277031349968, 'Thin Blue Line, The (1988)')),
 (1276, (4.088519233625427, 'Cool Hand Luke (1967)'))]

In [25]:
movie_rdd = movie_rdd.map(lambda line: (line[1][1],round(line[1][0],2)))

In [26]:
movie_rdd.take(100)

[('Spider-Man: Across the Spider-Verse (2023)', 4.21),
 ('Piper (2016)', 4.15),
 ('Jean de Florette (1986)', 4.1),
 ('Thin Blue Line, The (1988)', 4.09),
 ('Cool Hand Luke (1967)', 4.09),
 ('Rebecca (1940)', 4.06),
 ("Singin' in the Rain (1952)", 4.04),
 ('There Will Be Blood (2007)', 4.03),
 ('George Carlin: Back in Town (1996)', 4.03),
 ('Eat Drink Man Woman (Yin shi nan nu) (1994)', 4.03),
 ('Killer, The (Die xue shuang xiong) (1989)', 4.02),
 ('Graduate, The (1967)', 4.02),
 ('Killing, The (1956)', 4.01),
 ('Moon (2009)', 4.01),
 ('The Father (2020)', 4.0),
 ('The Lost Room (2006)', 3.99),
 ('Blood Simple (1984)', 3.98),
 ('Badlands (1973)', 3.98),
 ('Once Were Warriors (1994)', 3.98),
 ('Postman, The (Postino, Il) (1994)', 3.97),
 ("George Carlin: It's Bad for Ya! (2008)", 3.97),
 ('Big Heat, The (1953)', 3.96),
 ('Shall We Dance? (Shall We Dansu?) (1996)', 3.94),
 ("Won't You Be My Neighbor? (2018)", 3.94),
 ('City of Lost Children, The (Cité des enfants perdus, La) (1995)', 3.93

In [27]:
movie_info_with_genres =  movie_info.map(lambda line: (int(line[0]), (line[1], line[2].split('|'))))

In [28]:
movie_info_with_genres.take(10)

[(1,
  ('Toy Story (1995)',
   ['Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy'])),
 (2, ('Jumanji (1995)', ['Adventure', 'Children', 'Fantasy'])),
 (3, ('Grumpier Old Men (1995)', ['Comedy', 'Romance'])),
 (4, ('Waiting to Exhale (1995)', ['Comedy', 'Drama', 'Romance'])),
 (5, ('Father of the Bride Part II (1995)', ['Comedy'])),
 (6, ('Heat (1995)', ['Action', 'Crime', 'Thriller'])),
 (7, ('Sabrina (1995)', ['Comedy', 'Romance'])),
 (8, ('Tom and Huck (1995)', ['Adventure', 'Children'])),
 (9, ('Sudden Death (1995)', ['Action'])),
 (10, ('GoldenEye (1995)', ['Action', 'Adventure', 'Thriller']))]

In [29]:
movie_rdd_genres = movie_rdd_sorted.join(movie_info_with_genres)

In [30]:
movie_rdd_genres.take(10)

[(286897,
  (4.207413945278023,
   ('Spider-Man: Across the Spider-Verse (2023)',
    ['Action', 'Adventure', 'Animation', 'Sci-Fi']))),
 (160718, (4.15354535974974, ('Piper (2016)', ['Animation']))),
 (1131,
  (4.103291713961408, ('Jean de Florette (1986)', ['Drama', 'Mystery']))),
 (1189, (4.09277031349968, ('Thin Blue Line, The (1988)', ['Documentary']))),
 (1276, (4.088519233625427, ('Cool Hand Luke (1967)', ['Drama']))),
 (928,
  (4.0641265521347165,
   ('Rebecca (1940)', ['Drama', 'Mystery', 'Romance', 'Thriller']))),
 (899,
  (4.044915976148068,
   ("Singin' in the Rain (1952)", ['Comedy', 'Musical', 'Romance']))),
 (56782,
  (4.031560604104815, ('There Will Be Blood (2007)', ['Drama', 'Western']))),
 (136445,
  (4.031413612565445, ('George Carlin: Back in Town (1996)', ['Comedy']))),
 (232,
  (4.028991262907069,
   ('Eat Drink Man Woman (Yin shi nan nu) (1994)',
    ['Comedy', 'Drama', 'Romance'])))]

In [31]:
movie_rdd_genres_transformed = movie_rdd_genres.map(lambda line: (
    line[1][1][1],    # Title  (Index 0 of the inner tuple)
    round(line[1][0], 2)
))

# Now this should run without errors
movie_rdd_genres_transformed.take(10)

[(['Action', 'Adventure', 'Animation', 'Sci-Fi'], 4.21),
 (['Animation'], 4.15),
 (['Drama', 'Mystery'], 4.1),
 (['Documentary'], 4.09),
 (['Drama'], 4.09),
 (['Drama', 'Mystery', 'Romance', 'Thriller'], 4.06),
 (['Comedy', 'Musical', 'Romance'], 4.04),
 (['Drama', 'Western'], 4.03),
 (['Comedy'], 4.03),
 (['Comedy', 'Drama', 'Romance'], 4.03)]

In [32]:
movies_rdd_genres_flat = movie_rdd_genres_transformed.flatMap(lambda record: [(genre, (record[1], 1)) for genre in record[0]]) 

In [33]:
movies_rdd_genres_flat.take(20)

[('Action', (4.21, 1)),
 ('Adventure', (4.21, 1)),
 ('Animation', (4.21, 1)),
 ('Sci-Fi', (4.21, 1)),
 ('Animation', (4.15, 1)),
 ('Drama', (4.1, 1)),
 ('Mystery', (4.1, 1)),
 ('Documentary', (4.09, 1)),
 ('Drama', (4.09, 1)),
 ('Drama', (4.06, 1)),
 ('Mystery', (4.06, 1)),
 ('Romance', (4.06, 1)),
 ('Thriller', (4.06, 1)),
 ('Comedy', (4.04, 1)),
 ('Musical', (4.04, 1)),
 ('Romance', (4.04, 1)),
 ('Drama', (4.03, 1)),
 ('Western', (4.03, 1)),
 ('Comedy', (4.03, 1)),
 ('Comedy', (4.03, 1))]

In [34]:
movies_rdd_genres_reduced = movies_rdd_genres_flat.reduceByKey(lambda value1, value2: (value1[0] + value2[0], value1[1] + value2[1]))  

In [35]:
movies_rdd_genres_reduced.take(10)

[('Film-Noir', (433.7, 117)),
 ('Action', (6642.62, 2107)),
 ('Adventure', (4660.94, 1452)),
 ('IMAX', (521.7199999999999, 160)),
 ('Crime', (4917.409999999999, 1466)),
 ('Animation', (2364.29, 701)),
 ('Comedy', (13456.98, 4214)),
 ('Mystery', (2608.99, 779)),
 ('Musical', (1348.8000000000002, 402)),
 ('Children', (2452.89, 796))]

In [36]:
movie_rdd_genres_average = movies_rdd_genres_reduced.map(lambda line: (line[0],round(line[1][0]/line[1][1], 2)))

In [37]:
movie_rdd_genres_average.take(100)

[('Film-Noir', 3.71),
 ('Action', 3.15),
 ('Adventure', 3.21),
 ('IMAX', 3.26),
 ('Crime', 3.35),
 ('Animation', 3.37),
 ('Comedy', 3.19),
 ('Mystery', 3.35),
 ('Musical', 3.36),
 ('Children', 3.08),
 ('Thriller', 3.23),
 ('Sci-Fi', 3.14),
 ('Drama', 3.45),
 ('(no genres listed)', 3.45),
 ('Documentary', 3.65),
 ('Fantasy', 3.22),
 ('Western', 3.4),
 ('Horror', 2.98),
 ('Romance', 3.36),
 ('War', 3.54)]

In [38]:
moive_rdd_genres_sorted = movie_rdd_genres_average.sortBy(lambda line: line[1], False)

In [39]:
moive_rdd_genres_sorted.take(19)

[('Film-Noir', 3.71),
 ('Documentary', 3.65),
 ('War', 3.54),
 ('Drama', 3.45),
 ('(no genres listed)', 3.45),
 ('Western', 3.4),
 ('Animation', 3.37),
 ('Musical', 3.36),
 ('Romance', 3.36),
 ('Crime', 3.35),
 ('Mystery', 3.35),
 ('IMAX', 3.26),
 ('Thriller', 3.23),
 ('Fantasy', 3.22),
 ('Adventure', 3.21),
 ('Comedy', 3.19),
 ('Action', 3.15),
 ('Sci-Fi', 3.14),
 ('Children', 3.08)]